## 0. Введение

Этот ноутбук демонстрирует базовый запуск обучения детектора текста `EAST` в версии `0.1.10`. В примере показано, как использовать встроенные средства библиотеки `manuscript-ocr` для обучения собственной модели детекции на размеченных данных в формате COCO.

Для быстрого теста в примере может использоваться сокращенный сценарий обучения с минимальным числом эпох и небольшим размером батча, поэтому ноутбук предназначен прежде всего для демонстрации процесса запуска обучения.

Таким образом, библиотека `manuscript-ocr` предоставляет возможность обучать детекторы текста на собственных данных. Более подробная информация о параметрах обучения и доступных настройках приведена в полной документации проекта: https://konstantinkozhin.github.io/manuscript-ocr/0.1.10/en/api/detectors.html

Минимальные технические требования для запуска примера:

- Python 3.8+
- не менее 12 ГБ оперативной памяти
- NVIDIA GPU с объемом видеопамяти не менее 8 ГБ

Следует учитывать, что данный пример относится к обучению модели детекции, поэтому запуск на GPU является рекомендуемым сценарием. Запуск на CPU возможен, но будет существенно медленнее и не рассматривается как базовый вариант для апробации.


## 1. Установка зависимостей

In [ ]:
#может потребовать перезапуска среды в Google Colab
!pip install manuscript-ocr[dev]==0.1.10

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 959.8/959.8 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.3/69.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 110.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9

In [ ]:
# Затем обновите PyTorch на GPU версию, совместимую с вашей версией CUDA (например, для CUDA 11.8):
pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu118

## 2. Скачивание данных

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

from tqdm.auto import tqdm

base_url = "https://huggingface.co/datasets/anna4uonline/YeniseiGovReports-TD/resolve/main"
dataset_root = Path("datasets") / "YeniseiGovReports-TD"
dataset_root.mkdir(parents=True, exist_ok=True)

files = {
    "train_annotations": "train.json",
    "val_annotations": "test.json",
    "train_images_zip": "train_images.zip",
    "val_images_zip": "test_images.zip",
}

def download_with_progress(url: str, out_path: Path):
    if out_path.exists():
        print(f"Already exists: {out_path}")
        return

    last = [0]
    with tqdm(
        desc=f"Downloading {out_path.name}",
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
    ) as pbar:
        def reporthook(block_num, block_size, total_size):
            if total_size > 0:
                pbar.total = total_size
            downloaded = block_num * block_size
            pbar.update(downloaded - last[0])
            last[0] = downloaded

        urlretrieve(url, out_path, reporthook=reporthook)

def extract_zip_once(zip_path: Path, extract_to: Path):
    marker = extract_to / f".{zip_path.stem}.done"
    if marker.exists():
        print(f"Already extracted: {zip_path.name}")
        return

    print(f"Extracting {zip_path.name} ...")
    with ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)
    marker.touch()

def resolve_images_dir(root: Path, split_name: str) -> Path:
    direct = root / f"{split_name}_images"
    nested = direct / f"{split_name}_images"
    if nested.exists():
        return nested
    if direct.exists():
        return direct
    raise FileNotFoundError(f"Could not find extracted folder for {split_name}_images")

# Скачивание файлов датасета
for _, filename in files.items():
    download_with_progress(
        f"{base_url}/{filename}",
        dataset_root / filename,
    )

# Распаковка архивов с изображениями
extract_zip_once(dataset_root / "train_images.zip", dataset_root)
extract_zip_once(dataset_root / "test_images.zip", dataset_root)

# Пути для EAST.train(...)
train_images = str(resolve_images_dir(dataset_root, "train"))
train_annotations = str(dataset_root / "train.json")

# В датасете нет отдельного val split, поэтому для валидации используем test split
val_images = str(resolve_images_dir(dataset_root, "test"))
val_annotations = str(dataset_root / "test.json")

print("train_images      =", train_images)
print("train_annotations =", train_annotations)
print("val_images        =", val_images)
print("val_annotations   =", val_annotations)


Extracting train_images.zip ...
Extracting test_images.zip ...
train_images      = datasets/YeniseiGovReports-TD/train_images
train_annotations = datasets/YeniseiGovReports-TD/train.json
val_images        = datasets/YeniseiGovReports-TD/test_images
val_annotations   = datasets/YeniseiGovReports-TD/test.json


## 3. Запуск обучения

In [2]:
from manuscript.detectors import EAST

# Для быстрого теста используем test split и как train, и как val
train_images = val_images
train_annotations = val_annotations

EAST.train(
    train_images=train_images,
    train_anns=train_annotations,
    val_images=val_images,
    val_anns=val_annotations,
    epochs=1,
    batch_size=1,
)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 61.9MB/s]


Training configuration saved to: /content/experiments/resnet_quad/training_config.json


/usr/local/lib/python3.12/dist-packages/manuscript/detectors/_east/train_utils.py:249: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Train 1:   0%|          | 0/270 [00:00<?, ?it/s]

Attempting to export best model to ONNX...
Failed to export ONNX model: type object 'EAST' has no attribute 'export_to_onnx'


EASTModel(
  (backbone): ResNetFeatureExtractor(
    (extractor): ResNet(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Module(
        (0): Module(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_ru

## 4. Экспорт модели в формат для инференса

In [ ]:
from pathlib import Path

from manuscript.detectors import EAST

exp_dir = Path("experiments") / "resnet_quad"

EAST.export(
    weights_path=str(exp_dir / "checkpoints" / "best.pth"),
    output_path=str(exp_dir / "exported_model.pth"),
)

Loading checkpoint from C:\Users\pasha\manuscript-ocr\notebooks\experiments\resnet_quad\checkpoints\best.pth...
Auto-detecting backbone architecture...
   Detected 114 layer3 parameters
   Auto-detected backbone: resnet50

Loading PyTorch model...


C:\Users\pasha\manuscript-ocr\src\manuscript\detectors\_east\__init__.py:1091: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(str(weights_path), map_l

Loaded pretrained model from C:\Users\pasha\manuscript-ocr\notebooks\experiments\resnet_quad\checkpoints\best.pth
Model architecture: EASTWrapper
Input shape: torch.Size([1, 3, 1280, 1280])
Output shapes:
  - score_map: torch.Size([1, 1, 320, 320])
  - geo_map: torch.Size([1, 8, 320, 320])

Exporting to ONNX (opset 14)...
[OK] ONNX model saved to: C:\Users\pasha\manuscript-ocr\notebooks\experiments\resnet_quad\exported_model.pth

Verifying ONNX model...
[OK] ONNX model is valid

Simplifying ONNX model...
[OK] ONNX model simplified

Testing ONNX inference...
[OK] ONNX inference works!
  ONNX score_map shape: (1, 1, 320, 320)
  ONNX geo_map shape: (1, 8, 320, 320)

Comparing ONNX vs PyTorch outputs...
  Max difference in score_map: 0.000001
  Max difference in geo_map: 0.000017
  [OK] Outputs match!

[OK] Export complete! Model size: 107.5 MB

=== Summary ===
Backbone: resnet50
Input shape: [batch_size, 3, 1280, 1280]
Output shapes:
  - score_map: [batch_size, 1, H, W]
  - geo_map: [batc

Таким образом, `manuscript-ocr` предоставляет возможность обучать собственные модели детекции текста на пользовательских данных и экспортировать полученные веса для последующего использования.